In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import LSTM, Dense, Dropout, Embedding
from tensorflow.keras.models import Sequential
from sklearn.preprocessing import MinMaxScaler
import string, os

In [ ]:
#bài 1
# 1. Nạp dữ liệu CIFAR-10
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

# 2. Định nghĩa mô hình (Coi 32 hàng là 32 time steps, mỗi hàng 32*3 pixels)
model1 = Sequential([
    LSTM(128, input_shape=(32, 32 * 3), return_sequences=True),
    Dropout(0.2),
    LSTM(128),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')
])

# 3. Huấn luyện
# Reshape x_train: (50000, 32, 32, 3) -> (50000, 32, 96)
x_train_reshaped = x_train.reshape(-1, 32, 32 * 3)
model1.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model1.fit(x_train_reshaped, y_train, epochs=10, batch_size=64)
model1.save('lstm_cifar10.h5')

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 108s 134ms/step - accuracy: 0.3388 - loss: 1.8021
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 108s 138ms/step - accuracy: 0.4394 - loss: 1.5402
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 105s 135ms/step - accuracy: 0.4916 - loss: 1.4004
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 105s 134ms/step - accuracy: 0.5301 - loss: 1.3020
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 141s 133ms/step - accuracy: 0.5616 - loss: 1.2125
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 104s 133ms/step - accuracy: 0.5862 - loss: 1.1466
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 142s 133ms/step - accuracy: 0.6121 - loss: 1.0806
Epoch 8/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 104s 132ms/step - accuracy: 0.6349 - loss: 1.0261
Epoch 9/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 105s 134ms/step - accuracy: 0.6503 - loss: 0.9772
Epoch 10/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 105s 135ms/step - accuracy: 0.6674 - loss: 0.9273


In [ ]:
#bài 3
# 1. Nạp dữ liệu
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

# 2. Xây dựng mô hình tương tự mục 2.3 trong tài liệu
model3 = Sequential([
    LSTM(64, input_shape=(28, 28), return_sequences=True),
    Dropout(0.2),
    LSTM(64),
    Dense(32, activation='relu'),
    Dense(10, activation='softmax')
])

model3.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model3.fit(x_train, y_train, epochs=5)
model3.save('lstm_fashion.h5')

Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 58s 29ms/step - accuracy: 0.7752 - loss: 0.6175
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 51s 27ms/step - accuracy: 0.8453 - loss: 0.4222
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 51s 27ms/step - accuracy: 0.8599 - loss: 0.3757
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 82s 27ms/step - accuracy: 0.8707 - loss: 0.3452
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 82s 27ms/step - accuracy: 0.8787 - loss: 0.3242


In [ ]:
#câu 5
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.models import Sequential

# 1. Dữ liệu Truyện Kiều (Nguồn thay thế)
# Ân có thể thêm nhiều câu hơn để mô hình học tốt hơn
corpus = [
    "Trăm năm trong cõi người ta",
    "Chữ tài chữ mệnh khéo là ghét nhau",
    "Trải qua một cuộc bể dâu",
    "Những điều trông thấy mà đau đớn lòng",
    "Lạ gì bỉ sắc tư phong",
    "Trời xanh quen thói má hồng đánh ghen",
    "Cổ tay em trắng như ngà",
    "Đôi mắt em liếc như là dao cau"
]

# 2. Tokenization & n-gram
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)
total_words = len(tokenizer.word_index) + 1

input_sequences = []
for line in corpus:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

# 3. Padding sequences
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

# Chia tập predictors và label
X, y = input_sequences[:,:-1], input_sequences[:,-1]
y = to_categorical(y, num_classes=total_words)

# 4. Định nghĩa mô hình LSTM
model5 = Sequential([
    Embedding(total_words, 10, input_length=max_sequence_len-1),
    LSTM(100),
    Dropout(0.1),
    Dense(total_words, activation='softmax')
])

model5.compile(loss='categorical_crossentropy', optimizer='adam')
print("--- Đang huấn luyện mô hình sinh thơ... ---")
model5.fit(X, y, epochs=200, verbose=1) # Tăng epoch

# 5. Hàm sinh thơ
def generate_poetry(seed_text, next_words):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
        predicted = np.argmax(model5.predict(token_list, verbose=0), axis=-1)

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

# Test thử
print("\nKết quả sinh thơ:")
print(generate_poetry("Trăm năm", 4))

--- Đang huấn luyện mô hình sinh thơ... ---
Epoch 1/200


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - loss: 3.9705
Epoch 2/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 3.9681 
Epoch 3/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 3.9662
Epoch 4/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 3.9647
Epoch 5/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 3.9625
Epoch 6/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 3.9604
Epoch 7/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 3.9578
Epoch 8/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 3.9547
Epoch 9/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 3.9519
Epoch 10/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 3.9482
Epoch 11/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 3.9438
Epoch 12/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 3.9385
Epoch 13/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 3.9338
Epoch 14/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 3.9254
Epoch 15/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 3.9142
Epoch 16/200
2/2 ━━━━━━━━━━━━━

In [ ]:
#bài 7
import numpy as np
import tensorflow as tf
from flask import Flask, request, render_template_string
from PIL import Image
import io
import base64
from google.colab.output import eval_js

# 1. KHỞI TẠO FLASK
app = Flask(__name__)

# 2. NẠP MÔ HÌNH (Ví dụ nạp model Fashion-MNIST đã train ở Bài 3)
# Đảm bảo bạn đã chạy bài 3 và có file 'lstm_fashion.h5'
try:
    model = tf.keras.models.load_model('lstm_fashion.h5')
    labels = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
              "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
except:
    print("LƯU Ý: Không tìm thấy file model. Vui lòng chạy Bài 3 trước!")

# 3. GIAO DIỆN HTML (CSS & Layout)
html_layout = """
<!DOCTYPE html>
<html>
<head>
    <title>HUIT - LSTM Web App</title>
    <style>
        body { font-family: 'Segoe UI', sans-serif; background: #f0f2f5; display: flex; justify-content: center; padding: 50px; }
        .card { background: white; padding: 30px; border-radius: 15px; box-shadow: 0 4px 20px rgba(0,0,0,0.1); width: 400px; text-align: center; }
        .upload-zone { border: 2px dashed #3498db; padding: 20px; border-radius: 10px; background: #f8fbff; cursor: pointer; }
        .btn { background: #27ae60; color: white; border: none; padding: 12px 20px; border-radius: 5px; cursor: pointer; width: 100%; margin-top: 20px; font-weight: bold; }
        .result { margin-top: 20px; padding: 15px; background: #e8f5e9; border-radius: 8px; color: #2e7d32; font-size: 18px; font-weight: bold; }
        img { max-width: 100%; height: auto; border-radius: 5px; margin-top: 10px; }
    </style>
</head>
<body>
    <div class="card">
        <h2>LSTM Image Classifier</h2>
        <p>Phân loại thời trang (Fashion-MNIST)</p>
        <form method="post" enctype="multipart/form-data">
            <div class="upload-zone">
                <input type="file" name="file" accept="image/*" required>
            </div>
            <button type="submit" class="btn">PHÂN TÍCH ẢNH</button>
        </form>

        {% if result %}
            <div class="result">Dự đoán: {{ result }}</div>
            {% if img_data %}
                <img src="data:image/png;base64,{{ img_data }}">
            {% endif %}
        {% endif %}
    </div>
</body>
</html>
"""

# 4. LOGIC XỬ LÝ DỰ ĐOÁN
@app.route('/', methods=['GET', 'POST'])
def index():
    result = None
    img_base64 = None

    if request.method == 'POST':
        file = request.files.get('file')
        if file:
            # Đọc và Tiền xử lý ảnh (Resize về 28x28 grayscale cho Fashion-MNIST)
            image = Image.open(io.BytesIO(file.read())).convert('L').resize((28, 28))
            img_arr = np.array(image) / 255.0
            img_input = img_arr.reshape(1, 28, 28)

            # Dự đoán
            prediction = model.predict(img_input)
            class_idx = np.argmax(prediction)
            result = labels[class_idx]

            # Chuyển ảnh sang base64 để hiển thị lại trên web
            buf = io.BytesIO()
            image.save(buf, format="PNG")
            img_base64 = base64.b64encode(buf.getvalue()).decode('utf-8')

    return render_template_string(html_layout, result=result, img_data=img_base64)

# 5. CHẠY SERVER TRÊN COLAB
if __name__ == '__main__':
    # Tạo link công khai để mở web
    print(f"Bấm vào link này để mở giao diện Web: {eval_js('google.colab.kernel.proxyPort(5000)')}")
    app.run(port=5000)

Bấm vào link này để mở giao diện Web: https://5000-m-s-kkb-use1c1-3uzt70012mpkc-c.us-east1-1.prod.colab.dev
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [23/May/2026 04:29:24] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [23/May/2026 04:29:25] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [23/May/2026 04:29:54] "GET / HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 305ms/step


INFO:werkzeug:127.0.0.1 - - [23/May/2026 04:35:00] "POST / HTTP/1.1" 200 -
